# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdityaPrakash-Kaizu07/Flyrank-AI-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AdityaPrakash-Kaizu07/Flyrank-AI-intern"
REPO_DIR = "Flyrank-AI-intern"

if IN_COLAB:
    # In Colab, clone the repo if it doesn't exist yet
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

    # Install dependencies if you have a requirements.txt
    if os.path.exists("requirements.txt"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # On a local machine, find the repo root from wherever this notebook started
    # (Checking for 'README.md' or another unique folder/file at your repo root)
    while not os.path.isfile("README.md") and os.getcwd() != "/":
        os.chdir("..")
print("Working dir:", os.getcwd())
# Ensure we are actually at the repo root by asserting a known file/folder exists
assert os.path.exists("README.md"), "Root file not found — are you at the repo root?"
print("Starter data found. You're ready.")
os.makedirs('work/outputs', exist_ok=True)

Working dir: /content/Flyrank-AI-intern/Flyrank-AI-intern
Starter data found. You're ready.


In [9]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(df.shape)
print(df.columns.tolist())
print(df.head())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.0

In [10]:
# Step 1: Define your bucket edges (the ranges you want)
bucket_edges = [0, 30, 90, 180, float('inf')]
bucket_labels = ['0-30 days', '31-90 days', '91-180 days', '180+ days']

# Step 2: Use pd.cut() to assign each page to a bucket
df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=bucket_edges,
    labels=bucket_labels,
    right=False  # means [0, 30) not (0, 30]
)

# Step 3: Group and count
staleness_table = df['staleness_bucket'].value_counts().sort_index()
print("STALENESS SIGNAL")
print(staleness_table)
print(f"Total pages: {len(df)}")

STALENESS SIGNAL
staleness_bucket
0-30 days      20480
31-90 days       175
91-180 days     9171
180+ days        174
Name: count, dtype: int64
Total pages: 30000


In [11]:
# Step 1: Use pd.qcut() with q=4 (4 quartiles: top 25%, 50-75%, 25-50%, bottom 25%)
df['traffic_bucket'] = pd.qcut(
    df['impressions_90d'],
    q=4,
    labels=['Bottom 25%', '25-50%', '50-75%', 'Top 25%'],
    duplicates='drop'  # handles ties gracefully
)

# Step 2: Group and show n + mean impressions per bucket
traffic_table = df.groupby('traffic_bucket', observed=True).agg({
    'impressions_90d': ['count', 'mean', 'min', 'max']
}).round(1)

print("TRAFFIC SIGNAL")
print(traffic_table)

TRAFFIC SIGNAL
               impressions_90d                       
                         count     mean   min     max
traffic_bucket                                       
Bottom 25%                7503     19.7     1      81
25-50%                    7499    334.8    82     731
50-75%                    7498   1785.3   732    3615
Top 25%                   7500  18662.2  3616  517715


In [12]:
# AND logic
and_candidates = df[
    (df['days_since_last_update'] >= 91) &
    (df['traffic_bucket'] == 'Bottom 25%')
].shape[0]

# OR logic
or_candidates = df[
    (df['days_since_last_update'] >= 91) |
    (df['traffic_bucket'] == 'Bottom 25%')
].shape[0]

print(f"AND candidates: {and_candidates}")
print(f"OR candidates: {or_candidates}")

AND candidates: 1138
OR candidates: 15710


## 1. My rule and its reason codes

*Write the rule in plain words first.
Then the reason codes it can output.*
"Pages that are both 91+ days old AND in the bottom 25% of traffic over 90 days should be prioritized for refresh."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

REFRESH_PRIORITY: Both conditions met — old and low-traffic

   AGE_ONLY: Stale but still carries traffic — may be evergreen content
   
   TRAFFIC_ONLY: Invisible but new — may not have had time to mature
   
   NO_PRIORITY_SIGNAL: Recent and visible — don't touch

In [13]:
# Create binary flags
old = (df['days_since_last_update'] >= 91).astype(int)
low_traffic = (df['traffic_bucket'] == 'Bottom 25%').astype(int)

# Score: AND logic
df['baseline_score'] = old * low_traffic

# Reason codes
df['reason_code'] = df.apply(
    lambda row: (
        'REFRESH_PRIORITY' if row['baseline_score'] == 1
        else 'AGE_ONLY' if row['days_since_last_update'] >= 91
        else 'TRAFFIC_ONLY' if row['traffic_bucket'] == 'Bottom 25%'
        else 'NO_PRIORITY_SIGNAL'
    ),
    axis=1
)

# Action labels
df['action'] = df['reason_code'].apply(
    lambda code: {
        'REFRESH_PRIORITY': 'REFRESH',
        'AGE_ONLY': 'REVIEW',
        'TRAFFIC_ONLY': 'REVIEW',
        'NO_PRIORITY_SIGNAL': 'HOLD'
    }[code]
)

# Rank: by score (descending), then by staleness (descending) as tiebreaker
ranked = df.sort_values(
    by=['baseline_score', 'days_since_last_update'],
    ascending=[False, False]
).reset_index(drop=True)

# Write the ranked queue
ranked[['content_id', 'client_id', 'baseline_score', 'reason_code', 'action',
         'days_since_last_update', 'impressions_90d']].to_csv(
    'work/outputs/baseline_action_score.csv',
    index=False
)

print(f"Ranked queue written: {len(ranked)} rows")
print("\nTop 20 pages:")
print(ranked.head(20)[['content_id', 'baseline_score', 'reason_code', 'action',
                        'days_since_last_update', 'impressions_90d']])


Ranked queue written: 30000 rows

Top 20 pages:
              content_id  baseline_score       reason_code   action  \
0   content_3f3576c295f5               1  REFRESH_PRIORITY  REFRESH   
1   content_55a5b1c46474               1  REFRESH_PRIORITY  REFRESH   
2   content_f6fdf87348f6               1  REFRESH_PRIORITY  REFRESH   
3   content_8d56efff1e71               1  REFRESH_PRIORITY  REFRESH   
4   content_1b4ec72dafd4               1  REFRESH_PRIORITY  REFRESH   
5   content_f01216059a6a               1  REFRESH_PRIORITY  REFRESH   
6   content_e2b702f4f92b               1  REFRESH_PRIORITY  REFRESH   
7   content_06e19c6486b0               1  REFRESH_PRIORITY  REFRESH   
8   content_94991fe6268c               1  REFRESH_PRIORITY  REFRESH   
9   content_026a1e2a82fd               1  REFRESH_PRIORITY  REFRESH   
10  content_ccf25ed65a99               1  REFRESH_PRIORITY  REFRESH   
11  content_129753e3095f               1  REFRESH_PRIORITY  REFRESH   
12  content_f2b4acf220d9     

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
top_20 = ranked.head(20)[['content_id', 'days_since_last_update', 'impressions_90d',
                           'content_type', 'main_intent', 'word_count']].copy()

for idx, row in top_20.iterrows():
    print(f"{idx}. {row['content_id']}")
    print(f"   Days old: {row['days_since_last_update']}, Impressions: {row['impressions_90d']}")
    print(f"   Type: {row['content_type']}, Intent: {row['main_intent']}")
    print()


0. content_3f3576c295f5
   Days old: 373, Impressions: 1
   Type: keyword article, Intent: informational

1. content_55a5b1c46474
   Days old: 373, Impressions: 35
   Type: keyword article, Intent: informational

2. content_f6fdf87348f6
   Days old: 373, Impressions: 2
   Type: keyword article, Intent: informational

3. content_8d56efff1e71
   Days old: 372, Impressions: 1
   Type: keyword article, Intent: informational

4. content_1b4ec72dafd4
   Days old: 372, Impressions: 2
   Type: keyword article, Intent: informational

5. content_f01216059a6a
   Days old: 335, Impressions: 52
   Type: keyword article, Intent: nan

6. content_e2b702f4f92b
   Days old: 334, Impressions: 30
   Type: keyword article, Intent: nan

7. content_06e19c6486b0
   Days old: 334, Impressions: 10
   Type: keyword article, Intent: nan

8. content_94991fe6268c
   Days old: 313, Impressions: 7
   Type: keyword article, Intent: commercial

9. content_026a1e2a82fd
   Days old: 305, Impressions: 13
   Type: keyword 

Action: DEPRIORITIZE / INVESTIGATE | Why: 373 days old, 1 impression, informational | What would make it wrong: If the target keyword has meaningful search demand and the low visibility is caused by indexing or technical SEO issues.

Action: REFRESH | Why: 373 days old, 35 impressions, informational | What would make it wrong: If the keyword has little search demand or the existing page already matches search intent well.

Action: DEPRIORITIZE / INVESTIGATE | Why: 373 days old, 2 impressions, informational | What would make it wrong: If the page targets a valuable keyword but has a technical/indexing problem preventing visibility.

Action: DEPRIORITIZE / INVESTIGATE | Why: 372 days old, 1 impression, informational | What would make it wrong: If the keyword has strong demand and the page is simply ranking too poorly to receive impressions.

Action: DEPRIORITIZE / INVESTIGATE | Why: 372 days old, 2 impressions, informational | What would make it wrong: If the page targets a high-value topic and the low traffic is caused by ranking or indexing problems.

Action: INVESTIGATE BEFORE REFRESH | Why: 335 days old, 52 impressions, informational, missing intent | What would make it wrong: If missing intent means the page was never properly classified or its target keyword is unclear.

Action: INVESTIGATE BEFORE REFRESH | Why: 334 days old, 30 impressions, missing intent | What would make it wrong: If the page is abandoned or misconfigured and should be consolidated or removed instead.

Action: INVESTIGATE / DEPRIORITIZE | Why: 334 days old, 10 impressions, missing intent | What would make it wrong: If there is no clearly defined keyword or search intent to optimize the refresh around.

Action: INVESTIGATE KEYWORD STRATEGY | Why: 313 days old, 7 impressions, commercial intent | What would make it wrong: If the commercial keyword has little search demand or SERPs favor a different page type.

Action: INVESTIGATE KEYWORD STRATEGY | Why: 305 days old, 13 impressions, transactional intent | What would make it wrong: If users expect a product/service or conversion page rather than a keyword article.

Action: DEPRIORITIZE / INVESTIGATE | Why: 305 days old, 2 impressions, transactional intent | What would make it wrong: If the page is fundamentally the wrong content type for its transactional keyword and should be replaced rather than refreshed.

Action: INVESTIGATE KEYWORD STRATEGY | Why: 305 days old, 7 impressions, transactional intent | What would make it wrong: If the target query requires a product, landing, or comparison page rather than an informational article.

Action: REFRESH / TEST | Why: 305 days old, 17 impressions, transactional intent | What would make it wrong: If the keyword has weak demand or the SERP intent is dominated by pages that this article cannot realistically compete with.

Action: REFRESH | Why: 305 days old, 15 impressions, informational | What would make it wrong: If the topic has low search demand or the existing article already satisfies the relevant search intent.

Action: INVESTIGATE KEYWORD STRATEGY | Why: 305 days old, 5 impressions, transactional intent | What would make it wrong: If the page is targeting a transactional query where an article is the wrong format and a conversion-focused page is needed.

Action: INVESTIGATE KEYWORD STRATEGY | Why: 305 days old, 10 impressions, commercial intent | What would make it wrong: If the commercial query has low demand or competitors are ranking with a fundamentally different page type.

Action: REFRESH / TEST | Why: 305 days old, 13 impressions, informational | What would make it wrong: If the keyword has insufficient search volume to justify optimization effort.

Action: INVESTIGATE BEFORE REFRESH | Why: 304 days old, 11 impressions, missing intent | What would make it wrong: If missing intent reflects an abandoned or improperly configured page rather than merely incomplete metadata.

Action: DEPRIORITIZE / INVESTIGATE | Why: 304 days old, 1 impression, missing intent | What would make it wrong: If the page targets a valuable keyword and the single impression is caused by indexing, canonical, or technical problems.

Action: DEPRIORITIZE / INVESTIGATE | Why: 304 days old, 5 impressions, missing intent | What would make it wrong: If the page has a legitimate high-value target but its missing intent classification is simply a data-quality issue that can be corrected.

Overall skeptical conclusion: I would not treat all 20 as automatic refreshes. The baseline model is useful for prioritization, but the review reveals three different buckets:

1–2 impressions: first investigate whether the page should exist at all.
Missing intent: fix/classify the page before deciding what to refresh.
Commercial/transactional + low impressions: validate the keyword and SERP intent before investing in an article refresh.
Moderate impressions + clear informational intent: strongest candidates for an actual refresh.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
not_prioritized = ranked[ranked['baseline_score'] == 0].head(20)
print(not_prioritized[['content_id', 'days_since_last_update', 'impressions_90d', 'reason_code']])


                content_id  days_since_last_update  impressions_90d  \
1138  content_7a888d3d99c8                     313               95   
1139  content_6476d1d8c050                     313              304   
1140  content_02b0d6e30129                     313              176   
1141  content_f488400fca67                     305              155   
1142  content_d25a099b3726                     305              202   
1143  content_ab27c30d81f4                     304              103   
1144  content_df1fa766cac2                     304              206   
1145  content_07ce98c6085a                     304               85   
1146  content_4729b57ca036                     301              335   
1147  content_72496874f806                     301              821   
1148  content_7f116ae1f6f5                     301              954   
1149  content_4f241bad48a3                     236              285   
1150  content_e444c00065bd                     211              148   
1151  

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.